# NumPy 配套练习

先运行公共函数，再运行 12 道练习。每题的输出保存在对应单元格下方。

In [1]:
"""NumPy exercises NP-E1..E8 and NP-M1..M4.

Run this file to print all intermediate results. Calculations use full precision;
six decimal places are used only for display. Codex assisted implementation.
"""
import numpy as np


def _vector(value, name):
    result = np.asarray(value, dtype=np.float64)
    if result.ndim != 1 or result.size == 0 or not np.isfinite(result).all():
        raise ValueError(name + ' must be a finite, nonempty 1D vector')
    return result


def _nonzero_length(vector):
    length = np.sqrt(np.sum(vector * vector))
    if length == 0:
        raise ValueError('zero vector has no direction')
    return length


def angle_deg(a, b):
    """NP-E3: angle in degrees; zero vectors are invalid."""
    a, b = _vector(a, 'a'), _vector(b, 'b')
    if a.shape != b.shape:
        raise ValueError('vector dimensions must match')
    cosine = (a @ b) / (_nonzero_length(a) * _nonzero_length(b))
    return np.degrees(np.arccos(np.clip(cosine, -1.0, 1.0)))


def unit_vector(a):
    """NP-E5: a new unit vector; unlike the model helper, reject zero."""
    a = _vector(a, 'a')
    return a / _nonzero_length(a)


def _rows(query, rows):
    query = _vector(query, 'query')
    rows = np.asarray(rows, dtype=np.float64)
    if rows.ndim != 2 or rows.shape[1] != query.size or not np.isfinite(rows).all():
        raise ValueError('rows must be a finite 2D array matching query width')
    return query, rows, _nonzero_length(query)


def angles_to(query, X):
    """NP-M1: vectorized angles; zero rows retain their place as NaN."""
    query, X, query_length = _rows(query, X)
    lengths = np.sqrt(np.sum(X * X, axis=1))
    valid = lengths > 0
    result = np.full(X.shape[0], np.nan)
    cosine = (X[valid] @ query) / (lengths[valid] * query_length)
    result[valid] = np.degrees(np.arccos(np.clip(cosine, -1.0, 1.0)))
    return result


def top_k_cosine(query, gallery, k):
    """NP-M2: descending scores, stable ties, original gallery indices."""
    query, gallery, query_length = _rows(query, gallery)
    lengths = np.sqrt(np.sum(gallery * gallery, axis=1))
    valid_indices = np.flatnonzero(lengths > 0)
    if isinstance(k, (bool, np.bool_)) or not isinstance(k, (int, np.integer)):
        raise ValueError('k must be a positive integer')
    if not 1 <= k <= valid_indices.size:
        raise ValueError('k must not exceed the number of nonzero rows')
    scores = (gallery[valid_indices] @ query) / (lengths[valid_indices] * query_length)
    order = np.argsort(-scores, kind='stable')[:k]
    return valid_indices[order], scores[order]


def pairwise_distances(A, B):
    """NP-M3: [M,1,D] - [1,N,D] -> [M,N,D] -> [M,N]."""
    A, B = np.asarray(A, dtype=np.float64), np.asarray(B, dtype=np.float64)
    if A.ndim != 2 or B.ndim != 2 or A.shape[1] != B.shape[1]:
        raise ValueError('A and B must be 2D arrays with matching coordinate width')
    differences = A[:, None, :] - B[None, :, :]
    return np.sqrt(np.sum(differences * differences, axis=-1))


def _show(name, value):
    print(f'{name}: shape={np.shape(value)}\n{value}')


def _boundary_checks():
    # Boundaries required by the exercises: zeros, stable ties and invalid k.
    for call in (
        lambda: angle_deg([0, 0], [1, 0]),
        lambda: unit_vector([0, 0]),
        lambda: angles_to([0, 0], [[1, 0]]),
        lambda: top_k_cosine([0, 0], [[1, 0]], 1),
    ):
        try:
            call()
        except ValueError:
            pass
        else:
            raise AssertionError('zero direction must raise ValueError')
    gallery = np.array([[0., 0.], [1., 0.], [2., 0.], [-1., 0.]])
    indices, scores = top_k_cosine([1, 0], gallery, 3)
    np.testing.assert_array_equal(indices, [1, 2, 3])
    np.testing.assert_allclose(scores, [1, 1, -1])
    for k in (0, -1, 4, 1.5, True):
        try:
            top_k_cosine([1, 0], gallery, k)
        except ValueError:
            pass
        else:
            raise AssertionError('invalid k must raise ValueError')
    angles = angles_to([1, 0], [[0, 0], [1, 0], [-1, 0], [0, 1]])
    assert np.isnan(angles[0])
    np.testing.assert_allclose(angles[1:], [0, 180, 90], atol=1e-6)
    source = np.array([3., 4.])
    normalized = unit_vector(source)
    np.testing.assert_array_equal(source, [3., 4.])
    np.testing.assert_allclose(normalized, [.6, .8])
    np.testing.assert_allclose(pairwise_distances([[0, 0]], [[3, 4]]), [[5.]])
    print('\nRequired boundary checks: PASS')

np.set_printoptions(precision=6, suppress=True)

## NP-E1：数组与切片

In [2]:
# 整数索引 A[1] 取出一行并少一维；A[:, -2:] 用切片保留二维。
# 用 copy() 创建副本，再通过布尔索引将负数设为零。
print('NP-E1: arrays, shapes, slicing and a separate copy')
A = np.array([[.20, -.50, 1.10, .40], [.80, .30, -.70, .60], [-.40, .90, .10, -.20]])
print('A.ndim =', A.ndim, 'A.shape =', A.shape, 'A.size =', A.size)
_show('second row', A[1])
_show('last two columns', A[:, -2:])
B = A.copy()
B[B < 0] = 0
_show('B', B)
_show('unchanged A', A)

NP-E1: arrays, shapes, slicing and a separate copy
A.ndim = 2 A.shape = (3, 4) A.size = 12
second row: shape=(4,)
[ 0.8  0.3 -0.7  0.6]
last two columns: shape=(3, 2)
[[ 1.1  0.4]
 [-0.7  0.6]
 [ 0.1 -0.2]]
B: shape=(3, 4)
[[0.2 0.  1.1 0.4]
 [0.8 0.3 0.  0.6]
 [0.  0.9 0.1 0. ]]
unchanged A: shape=(3, 4)
[[ 0.2 -0.5  1.1  0.4]
 [ 0.8  0.3 -0.7  0.6]
 [-0.4  0.9  0.1 -0.2]]


## NP-E2：逐元素乘积与点积

In [3]:
# * 将对应位置相乘；一维数组用 @ 做点乘，结果是标量。
# 向量长度等于所有元素的平方和开根号。
print('\nNP-E2: elementwise product, dot product and lengths')
a, b = np.array([.80, -.60, .70]), np.array([-.20, .90, .40])
_show('a * b', a * b)
_show('a @ b', a @ b)
for name, vector in (('a', a), ('b', b)):
    length = np.sqrt(np.sum(vector * vector))
    print(name, 'length =', length)
    np.testing.assert_allclose(length, np.linalg.norm(vector), atol=1e-6)


NP-E2: elementwise product, dot product and lengths
a * b: shape=(3,)
[-0.16 -0.54  0.28]
a @ b: shape=()
-0.4200000000000001
a length = 1.2206555615733703
b length = 1.0049875621120892


## NP-E3：向量夹角

In [4]:
# 用点积除以两个向量长度得到余弦值，再求反余弦并转成度。
# 公共函数会把余弦限制在 [-1, 1]，并拒绝零向量。
print('\nNP-E3: angle')
_show('degrees', angle_deg([1., -1.], [1., 0.]))


NP-E3: angle
degrees: shape=()
45.00000000000001


## NP-E4：距离计算

In [5]:
# 先用 p - q 求坐标差。欧氏距离是平方和开根号，曼哈顿距离是绝对值之和。
print('\nNP-E4: distances')
p, q = np.array([-.50, 1.20, 2.00]), np.array([.80, -.40, 1.10])
_show('p - q', p - q)
_show('Euclidean', np.sqrt(np.sum((p - q) ** 2)))
_show('Manhattan', np.sum(np.abs(p - q)))


NP-E4: distances
p - q: shape=(3,)
[-1.3  1.6  0.9]
Euclidean: shape=()
2.249444375840399
Manhattan: shape=()
3.8000000000000003


## NP-E5：单位向量

In [6]:
# 每个元素除以同一个向量长度，得到新数组。用 allclose 检查长度是否接近 1。
print('\nNP-E5: unit vector')
a = np.array([-1.20, .40, .80])
normalized = unit_vector(a)
_show('unit vector', normalized)
print('unit length:', np.allclose(np.linalg.norm(normalized), 1.0))


NP-E5: unit vector
unit vector: shape=(3,)
[-0.801784  0.267261  0.534522]
unit length: True


## NP-E6：沿轴汇总

In [7]:
# axis=0 消去行轴，每列得到一个和；axis=1 消去列轴，每行得到一个均值。
print('\nNP-E6: reductions')
A = np.array([[.20, -.50, 1.10], [.80, .40, -.30]])
_show('sum(axis=0), one sum per column', A.sum(axis=0))
_show('mean(axis=1), one mean per row', A.mean(axis=1))
_show('overall maximum', A.max())


NP-E6: reductions
sum(axis=0), one sum per column: shape=(3,)
[ 1.  -0.1  0.8]
mean(axis=1), one mean per row: shape=(2,)
[0.266667 0.3     ]
overall maximum: shape=()
1.1


## NP-E7：广播

In [8]:
# scale 和 bias 都从末轴与 X 的列对齐，三列分别使用各自的系数和偏置。
print('\nNP-E7: scale and bias broadcast across rows')
X = np.array([[.80, -.60, .70], [-.40, .50, .20]])
scale, bias = np.array([1.50, .50, -1.00]), np.array([-.10, .20, .30])
_show('Y = X * scale + bias', X * scale + bias)


NP-E7: scale and bias broadcast across rows
Y = X * scale + bias: shape=(2, 3)
[[ 1.1  -0.1  -0.4 ]
 [-0.7   0.45  0.1 ]]


## NP-E8：图像批次

In [9]:
# stack 新增批次轴；下标 2 选出蓝色通道。
# transpose(0, 2, 3, 1) 将通道轴移到最后。
print('\nNP-E8: image batch')
a = np.array([[[.20, -.40], [.70, .10]], [[.30, .80], [-.20, .50]], [[-.10, .60], [.40, -.30]]])
b = a + .10
batch = np.stack((a, b), axis=0)
print('BCHW:', batch.shape)
_show('blue channels', batch[:, 2, :, :])
print('BHWC:', batch.transpose(0, 2, 3, 1).shape)


NP-E8: image batch
BCHW: (2, 3, 2, 2)
blue channels: shape=(2, 2, 2)
[[[-0.1  0.6]
  [ 0.4 -0.3]]

 [[ 0.   0.7]
  [ 0.5 -0.2]]]
BHWC: (2, 2, 2, 3)


## NP-M1：批量向量夹角

In [10]:
# 用矩阵乘法同时算出各行与查询向量的点积。零行没有方向，结果留为 NaN。
print('\nNP-M1: angles to all rows')
query = np.array([.60, -.80, .00])
X = np.array([[.30, -.40, .00], [-.60, .80, .00], [.80, .60, .00], [.00, .00, .00]])
_show('angles', angles_to(query, X))


NP-M1: angles to all rows
angles: shape=(4,)
[  0. 180.  90.  nan]


## NP-M2：余弦相似度 Top-k

In [11]:
# 按余弦相似度从高到低选 k 项。同分保留原顺序，返回的是原始行下标。
print('\nNP-M2: stable top-k cosine retrieval')
query = np.array([.60, -.80, .00])
gallery = np.array([[.60, -.80, .00], [.30, -.40, .00], [.80, .60, .00], [-.60, .80, .00], [.00, .00, .00]])
ids = np.array(['img_a', 'img_b', 'img_c', 'img_d', 'img_zero'])
indices, similarities = top_k_cosine(query, gallery, 3)
_show('indices', indices)
_show('similarities', similarities)
_show('selected IDs', ids[indices])


NP-M2: stable top-k cosine retrieval
indices: shape=(3,)
[0 1 2]
similarities: shape=(3,)
[1. 1. 0.]
selected IDs: shape=(3,)
['img_a' 'img_b' 'img_c']


## NP-M3：两两距离矩阵

In [12]:
# A[:, None, :] 和 B[None, :, :] 通过广播形成两两坐标差。
# 距离矩阵每行对应 A 中的一个点，argmin(axis=1) 找对应的最近点。
print('\nNP-M3: all pairwise distances')
A, B = np.array([[.20, -.40], [.80, .30], [-.50, .70]]), np.array([[.10, .20], [-.60, -.20]])
print('broadcast difference shape:', (A[:, None, :] - B[None, :, :]).shape)
distances = pairwise_distances(A, B)
_show('D', distances)
_show('nearest B indices', distances.argmin(axis=1))


NP-M3: all pairwise distances
broadcast difference shape: (3, 2, 2)
D: shape=(3, 2)
[[0.608276 0.824621]
 [0.707107 1.486607]
 [0.781025 0.905539]]
nearest B indices: shape=(3,)
[0 0 0]


## NP-M4：两层前向计算

In [13]:
# 先算 Z = X @ W1.T + b1，用 ReLU 把负数变成 0，再乘 W2 并加 b2。
# 两组权重按 [输出维度, 输入维度] 存储，所以计算时转置。
print('\nNP-M4: linear -> ReLU -> linear (provided parameters)')
X = np.array([[.80, -.60, .70], [-.40, -.50, .90]])
W1, b1 = np.array([[-.10, .20, -.80], [.30, -.50, .10]]), np.array([-.15, .05])
W2, b2 = np.array([[.20, -.60], [-.80, .52]]), np.array([.10, -.20])
Z = X @ W1.T + b1
H = np.maximum(Z, 0.0)
Y = H @ W2.T + b2
for name, value in (('Z', Z), ('H', H), ('Y', Y)):
    _show(name, value)


NP-M4: linear -> ReLU -> linear (provided parameters)
Z: shape=(2, 2)
[[-0.91  0.66]
 [-0.93  0.27]]
H: shape=(2, 2)
[[0.   0.66]
 [0.   0.27]]
Y: shape=(2, 2)
[[-0.296   0.1432]
 [-0.062  -0.0596]]


## 边界检查

In [14]:
_boundary_checks()


Required boundary checks: PASS
